In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"C:\Users\vishwa\Downloads\archive (18)\framingham.csv")

In [ ]:
df

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
for i in df:
    print("*"*10,i,"*"*10)
    print()
    print(df[i].unique())
    print()

In [ ]:
df[df.duplicated()]

In [ ]:
df.describe()

In [ ]:
df.select_dtypes(include = ["number","float"]).columns

In [ ]:
df.select_dtypes(include = "string")

In [ ]:
median_cols = ['glucose', 'totChol', 'BMI', 'heartRate', 'cigsPerDay']

for col in median_cols:
    print(f"{col}: skewness = {df[col].skew():.2f}")

### No Feature Enginnering is required for the dataset.
- Discrete or categorical values are ready in numerical only
- But for all continuous or numerical the feature engineering is not required as per the decision tree

In [ ]:
X = df[['male', 'age', 'education', 'currentSmoker', 'cigsPerDay', 'BPMeds',
       'prevalentStroke', 'prevalentHyp', 'diabetes', 'totChol', 'sysBP',
       'diaBP', 'BMI', 'heartRate', 'glucose']]

In [ ]:
y = df[['TenYearCHD']]

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,train_size=0.8,random_state = 42)

In [ ]:
X_train

In [ ]:
y_train

In [ ]:
X_test

# Before Hyperparametric Tunning

In [ ]:
from sklearn.tree import DecisionTreeClassifier
DT1 = DecisionTreeClassifier()
DT1.fit(X_train,y_train)

In [ ]:
import pandas as pd

y_pred1 = DT1.predict(X_test)

y_pred1 = pd.DataFrame(
    y_pred1,
    columns=['TenYearCHD']
)

y_pred1

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

models = {
    "Model 1": model1,
    "Model 2": model2,
    "Model 3": model3
}

for name, model in models.items():

    # Predictions
    y_pred = model.predict(X_test_trans)

    # Metrics
    accuracy = accuracy_score(y_test_trans, y_pred)
    precision = precision_score(y_test_trans, y_pred, average='weighted')
    recall = recall_score(y_test_trans, y_pred, average='weighted')
    f1 = f1_score(y_test_trans, y_pred, average='weighted')

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test_trans, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test_trans, y_pred))

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [ ]:
# Metrics
accuracy = accuracy_score(y_test, y_pred1)
precision = precision_score(y_test, y_pred1, average='weighted')
recall = recall_score(y_test, y_pred1, average='weighted')
f1 = f1_score(y_test, y_pred1, average='weighted')

In [ ]:
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred1))

print("\nClassification Report:")
print(classification_report(y_test, y_pred1))

# After the Hyperparameter Tunning

### Grid CV

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth'        : [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
    'criterion'        : ['gini', 'entropy']
}

grid_search = GridSearchCV(
    estimator  = DecisionTreeClassifier(random_state=42),
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'roc_auc',
    n_jobs     = -1,
    verbose    = 1
)

grid_search.fit(X_train, y_train)

print("Best Parameters :", grid_search.best_params_)
print("Best ROC-AUC    :", round(grid_search.best_score_, 4))

DT_grid = grid_search.best_estimator_

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'max_depth'        : [3, 5, 7, 10, 15, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 4, 8],
    'criterion'        : ['gini', 'entropy'],
    'max_features'     : ['sqrt', 'log2', None]
}

random_search = RandomizedSearchCV(
    estimator            = DecisionTreeClassifier(random_state=42),
    param_distributions  = param_dist,
    n_iter               = 40,
    cv                   = 5,
    scoring              = 'roc_auc',
    random_state         = 42,
    n_jobs               = -1,
    verbose              = 1
)

random_search.fit(X_train, y_train)

print("Best Parameters :", random_search.best_params_)
print("Best ROC-AUC    :", round(random_search.best_score_, 4))

DT_random = random_search.best_estimator_

In [ ]:
import optuna
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import KFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

# 1. Create Objective Function
def Objective(trial):
    # create search space
    max_depth         = trial.suggest_int('max_depth', 2, 20)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf  = trial.suggest_int('min_samples_leaf', 1, 10)
    criterion         = trial.suggest_categorical('criterion', ['gini', 'entropy'])

    # train the algorithm
    dt = DecisionTreeClassifier(
        max_depth         = max_depth,
        min_samples_split = min_samples_split,
        min_samples_leaf  = min_samples_leaf,
        criterion         = criterion,
        random_state      = 42
    )

    # get the Performance metrics
    kf    = KFold(n_splits=5, shuffle=True, random_state=23)
    score = cross_val_score(estimator=dt, X=X_train, y=y_train,
                            scoring='roc_auc', cv=kf).mean()
    return score

# 2. Create the Study
study = optuna.create_study(direction='maximize')

# 3. Evaluate model Performance
study.optimize(Objective, n_trials=40, show_progress_bar=True)

In [ ]:
print("Best Parameters :", study.best_params)
print("Best ROC-AUC    :", round(study.best_value, 4))

# Build the best Bayesian model
DT_bayesian = DecisionTreeClassifier(**study.best_params, random_state=42)
DT_bayesian.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score, accuracy_score

models = {
    'DT Baseline'    : DT1,
    'DT GridSearch'  : DT_grid,
    'DT RandomSearch': DT_random,
    'DT Bayesian'    : DT_bayesian
}

print(f"{'Model':<20} {'Accuracy':>10} {'Recall':>10} {'F1':>10} {'ROC-AUC':>10}")
print("-" * 62)

for name, model in models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc     = accuracy_score(y_test, y_pred)
    rec     = recall_score(y_test, y_pred)
    f1      = f1_score(y_test, y_pred)
    roc     = roc_auc_score(y_test, y_prob)

    print(f"{name:<20} {acc:>10.4f} {rec:>10.4f} {f1:>10.4f} {roc:>10.4f}")

# FEATURE SELECTION

Three categories of feature selection applied to the Framingham dataset:
- **Filter Methods** — statistical tests independent of any model
- **Wrapper Methods** — model-based selection (RFE)
- **Embedded Methods** — feature importance from the trained Decision Tree

> Note: SelectKBest, mutual_info_classif, RFE, and VarianceThreshold cannot handle NaN.
> A SimpleImputer is applied to create `X_train_imp` / `X_test_imp` for these methods only.
> The Decision Tree models above continue to use the original `X_train` (DT handles NaN natively).

In [ ]:
from sklearn.impute import SimpleImputer

median_cols = ['glucose', 'totChol', 'BMI', 'heartRate', 'cigsPerDay']
mode_cols   = ['BPMeds', 'education']

imp_median = SimpleImputer(strategy='median')
imp_mode   = SimpleImputer(strategy='most_frequent')

X_train_imp = X_train.copy()
X_test_imp  = X_test.copy()

X_train_imp[median_cols] = imp_median.fit_transform(X_train[median_cols])
X_test_imp[median_cols]  = imp_median.transform(X_test[median_cols])

X_train_imp[mode_cols]   = imp_mode.fit_transform(X_train[mode_cols])
X_test_imp[mode_cols]    = imp_mode.transform(X_test[mode_cols])

print("NaN remaining in X_train_imp:", X_train_imp.isnull().sum().sum())

## Filter Methods
### 1.1 Zero / Low Variance Features — VarianceThreshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.1)
vt.fit(X_train_imp)

selected_vt  = list(vt.get_feature_names_out())
removed_vt   = [c for c in X_train_imp.columns if c not in selected_vt]

print(f"Features kept   ({len(selected_vt)}): {selected_vt}")
print(f"Features removed ({len(removed_vt)}): {removed_vt}")

### 1.2 SelectKBest — ANOVA F-test (f_classif)

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

kbest = SelectKBest(f_classif, k=10)
kbest.fit(X_train_imp, y_train.values.ravel())

scores_df = pd.DataFrame({
    'Feature': X_train_imp.columns,
    'F_Score': kbest.scores_
}).sort_values('F_Score', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=scores_df, x='Feature', y='F_Score', palette='Blues_r')
plt.xticks(rotation=45, ha='right')
plt.title('SelectKBest — ANOVA F-Scores (higher = more relevant)')
plt.tight_layout()
plt.show()

print("\nTop 10 features selected:", list(kbest.get_feature_names_out()))

### 1.3 Mutual Information

In [ ]:
from sklearn.feature_selection import mutual_info_classif

mi_scores = mutual_info_classif(X_train_imp, y_train.values.ravel(), random_state=42)

mi_df = pd.DataFrame({
    'Feature' : X_train_imp.columns,
    'MI_Score': mi_scores
}).sort_values('MI_Score', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=mi_df, x='Feature', y='MI_Score', palette='Greens_r')
plt.xticks(rotation=45, ha='right')
plt.title('Mutual Information Scores (higher = more dependent on target)')
plt.tight_layout()
plt.show()

print(mi_df.to_string(index=False))

## Wrapper Methods
### 2.1 RFE — Recursive Feature Elimination

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

rfe = RFE(
    estimator            = DecisionTreeClassifier(random_state=42),
    n_features_to_select = 10
)
rfe.fit(X_train_imp, y_train.values.ravel())

selected_rfe = X_train_imp.columns[rfe.support_].tolist()
ranking_df   = pd.DataFrame({
    'Feature': X_train_imp.columns,
    'Rank'   : rfe.ranking_
}).sort_values('Rank')

print("Features selected by RFE:", selected_rfe)
print()
print(ranking_df.to_string(index=False))

## Embedded Methods
### 3.1 Decision Tree Feature Importances

In [ ]:
# DT_bayesian was fitted on X_train — Decision Tree computes feature importances
# natively during training (no separate selection step needed)

fi_df = pd.DataFrame({
    'Feature'   : X_train.columns,
    'Importance': DT_bayesian.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=fi_df, x='Feature', y='Importance', palette='Oranges_r')
plt.xticks(rotation=45, ha='right')
plt.title('Feature Importances — DT Bayesian Best Model (Embedded Method)')
plt.ylabel('Gini Importance')
plt.tight_layout()
plt.show()

print(fi_df.to_string(index=False))

# CROSS VALIDATION

Five cross-validation strategies applied using `DT_bayesian` (best model from Bayesian Optimization).
Scoring metric: `roc_auc` (most informative for imbalanced CHD data — 15% positive class).

| Strategy | Best For |
|---|---|
| K-Fold | General purpose |
| Stratified K-Fold | Imbalanced classes (recommended here) |
| LOOCV | Small datasets |
| Leave-P-Out | Educational — impractical on large data |

## 1. K-Fold (5 splits)

In [ ]:
from sklearn.model_selection import KFold, cross_val_score

k5 = KFold(n_splits=5, shuffle=True, random_state=44)

scores_k5 = cross_val_score(
    estimator = DT_bayesian,
    X         = X_train,
    y         = y_train.values.ravel(),
    cv        = k5,
    scoring   = 'roc_auc'
)

print("KFold (5 splits) — ROC-AUC per fold:", scores_k5.round(4))
print("Mean ROC-AUC :", scores_k5.mean().round(4))
print("Std  ROC-AUC :", scores_k5.std().round(4))

## 2. K-Fold (10 splits)

In [ ]:
k10 = KFold(n_splits=10, shuffle=True, random_state=44)

scores_k10 = cross_val_score(
    estimator = DT_bayesian,
    X         = X_train,
    y         = y_train.values.ravel(),
    cv        = k10,
    scoring   = 'roc_auc'
)

print("KFold (10 splits) — ROC-AUC per fold:", scores_k10.round(4))
print("Mean ROC-AUC :", scores_k10.mean().round(4))
print("Std  ROC-AUC :", scores_k10.std().round(4))

## 3. Stratified K-Fold (5 splits) — Recommended

Most appropriate here because TenYearCHD has 15% positive rate (class imbalance).
Stratified K-Fold preserves that 15/85 ratio in every fold.

In [ ]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=44)

scores_skf = cross_val_score(
    estimator = DT_bayesian,
    X         = X_train,
    y         = y_train.values.ravel(),
    cv        = skf,
    scoring   = 'roc_auc'
)

print("StratifiedKFold (5 splits) — ROC-AUC per fold:", scores_skf.round(4))
print("Mean ROC-AUC :", scores_skf.mean().round(4))
print("Std  ROC-AUC :", scores_skf.std().round(4))

## 4. Leave One Out CV (LOOCV)

Trains n models (one per sample). Each model trains on n-1 samples, tests on 1.
Slow on large data — included for educational completeness.

In [ ]:
from sklearn.model_selection import LeaveOneOut

loocv = LeaveOneOut()

scores_loocv = cross_val_score(
    estimator = DT_bayesian,
    X         = X_train,
    y         = y_train.values.ravel(),
    cv        = loocv,
    scoring   = 'accuracy',
    n_jobs    = -1   # parallel to speed up
)

print("LOOCV Mean Accuracy:", scores_loocv.mean().round(4))
print("LOOCV Std  Accuracy:", scores_loocv.std().round(4))

## 5. Leave P Out CV (LPOCV)

Leaves p samples out as test set in each fold — creates C(n, p) folds.
Included to match reference notebook. With p=2: C(3392, 2) ≈ 5.7M folds — still slow.

In [ ]:
from sklearn.model_selection import LeavePOut

lpov = LeavePOut(p=2)

scores_lpov = cross_val_score(
    estimator = DT_bayesian,
    X         = X_train,
    y         = y_train.values.ravel(),
    cv        = lpov,
    scoring   = 'accuracy',
    n_jobs    = -1
)

print("LeavePOut (p=2) Mean Accuracy:", scores_lpov.mean().round(4))
print("LeavePOut (p=2) Std  Accuracy:", scores_lpov.std().round(4))

## Cross Validation Summary

In [ ]:
cv_summary = pd.DataFrame({
    'Strategy'    : ['KFold-5', 'KFold-10', 'StratifiedKFold-5', 'LOOCV'],
    'Metric'      : ['ROC-AUC', 'ROC-AUC', 'ROC-AUC', 'Accuracy'],
    'Mean Score'  : [scores_k5.mean(), scores_k10.mean(), scores_skf.mean(), scores_loocv.mean()],
    'Std Score'   : [scores_k5.std(),  scores_k10.std(),  scores_skf.std(),  scores_loocv.std()]
})

cv_summary['Mean Score'] = cv_summary['Mean Score'].round(4)
cv_summary['Std Score']  = cv_summary['Std Score'].round(4)

print(cv_summary.to_string(index=False))

# BIAS-VARIANCE TRADEOFF

Two approaches following the reference notebook:
1. **sklearn learning_curve** — manual plot with confidence bands (train vs validation)
2. **mlxtend bias_variance_decomp** — numerical decomposition of expected loss into bias² + variance
3. **mlxtend plot_learning_curves** — quick visual (matches reference notebook style)

Uses `X_train_imp` / `X_test_imp` (imputed) since mlxtend does not handle NaN internally.

## 1. Learning Curves — sklearn (Train vs Validation)

In [ ]:
from sklearn.model_selection import learning_curve
import numpy as np

dt_for_lc = DecisionTreeClassifier(**study.best_params, random_state=42)

train_sizes, train_scores, val_scores = learning_curve(
    estimator    = dt_for_lc,
    X            = X_train_imp,
    y            = y_train.values.ravel(),
    cv           = 5,
    scoring      = 'roc_auc',
    train_sizes  = np.linspace(0.1, 1.0, 10),
    n_jobs       = -1,
    random_state = 42
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='royalblue',  label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                 alpha=0.15, color='royalblue')
plt.plot(train_sizes, val_mean,   'o-', color='seagreen',   label='Validation Score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                 alpha=0.15, color='seagreen')
plt.xlabel('Training Samples')
plt.ylabel('ROC-AUC Score')
plt.title('Learning Curves — Bias-Variance Tradeoff (Decision Tree)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Bias-Variance Decomposition — mlxtend

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

dt_for_bv = DecisionTreeClassifier(**study.best_params, random_state=42)

avg_expected_loss, avg_bias, avg_var = bias_variance_decomp(
    dt_for_bv,
    X_train_imp.values,
    y_train.values.ravel(),
    X_test_imp.values,
    y_test.values.ravel(),
    loss        = '0-1_loss',
    random_seed = 42
)

print('=' * 42)
print('  Bias-Variance Decomposition Results')
print('=' * 42)
print(f'  Average Expected Loss : {avg_expected_loss:.4f}')
print(f'  Average Bias²         : {avg_bias:.4f}')
print(f'  Average Variance      : {avg_var:.4f}')
print('-' * 42)
print(f'  Bias² + Variance      : {avg_bias + avg_var:.4f}')
print('=' * 42)

## 3. Learning Curves — mlxtend (reference notebook style)

In [ ]:
from mlxtend.plotting import plot_learning_curves

dt_for_lc2 = DecisionTreeClassifier(**study.best_params, random_state=42)

plot_learning_curves(
    X_train_imp.values,
    y_train.values.ravel(),
    X_test_imp.values,
    y_test.values.ravel(),
    clf = dt_for_lc2
)
plt.title('Learning Curves — Decision Tree (mlxtend)')
plt.tight_layout()
plt.show()